# 01_c_add_features

> Purpose: extend the preprocessed dataset with feature engineering that helps segmentation and modeling.

This notebook loads `data/processed/preprocessed_data.csv`, creates derived features for customer profile and channel behavior, then saves an enriched dataset to `data/features_added/preprocessed_data_features_added.csv`.

## Step 1) Load preprocessed input data

> Reads `data/processed/preprocessed_data.csv`, validates file availability, and displays a quick preview for sanity checking.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'

input_path = DATA_DIR / 'processed' / 'preprocessed_data.csv'
if not input_path.exists():
    raise FileNotFoundError(f'Input file not found: {input_path}')

df = pd.read_csv(input_path)
print('Loaded:', input_path)
print('Shape:', df.shape)
df.head()

Loaded: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\processed\preprocessed_data.csv
Shape: (39826, 20)


,person,offer,offer_completed_event,offer_received,offer_viewed,age,income,reward,difficulty,duration,offer_completed,days_since_registration,web,email,mobile,social,gender_M,gender_O,gender_U,offer_type_discount
0,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,True,True,True,33.0,72000.0,5,5,5,1,461,1,1,1,1,True,False,False,False
1,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,True,True,True,33.0,72000.0,2,10,10,1,461,1,1,1,1,True,False,False,True
2,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,False,True,True,55.0,64000.0,5,5,5,0,92,1,1,1,1,False,False,True,False
3,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,True,True,True,40.0,57000.0,5,20,10,1,198,1,1,0,0,False,True,False,True
4,0011e0d4e6b944f998e987f904e8c1e5,2298d6c36e964ae4a3e7e9706d1fb8c2,True,True,True,40.0,57000.0,3,7,7,1,198,1,1,1,1,False,True,False,True


## Step 2) Engineer additional features

> Adds `membership_duration_months`, creates `age_group` and `income_group`, and builds channel availability flags from existing channel columns (`web`, `email`, `mobile`, `social`).

In [2]:
# Membership duration in months
if 'days_since_registration' not in df.columns:
    raise KeyError("Expected column 'days_since_registration' in input data")

df['membership_duration_months'] = (df['days_since_registration'] / 30.44).round(2)

# Age groups
if 'age' not in df.columns:
    raise KeyError("Expected column 'age' in input data")

age_bins = [0, 25, 35, 45, 55, 65, np.inf]
age_labels = ['18-25', '26-35', '36-45', '46-55', '56-65', '66+']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, include_lowest=True)

# Income groups
if 'income' not in df.columns:
    raise KeyError("Expected column 'income' in input data")

income_bins = [0, 40000, 60000, 80000, 100000, np.inf]
income_labels = ['low', 'lower_middle', 'middle', 'upper_middle', 'high']
df['income_group'] = pd.cut(df['income'], bins=income_bins, labels=income_labels, include_lowest=True)

# Channel flags from existing channel indicator columns
for channel_col in ['web', 'email', 'mobile', 'social']:
    if channel_col not in df.columns:
        raise KeyError(f"Expected channel column '{channel_col}' in input data")

df['has_web_channel'] = (df['web'] > 0).astype(int)
df['has_email_channel'] = (df['email'] > 0).astype(int)
df['has_mobile_channel'] = (df['mobile'] > 0).astype(int)
df['has_social_channel'] = (df['social'] > 0).astype(int)

print('Feature engineering completed.')
print('New columns added:', [
    'membership_duration_months', 'age_group', 'income_group',
    'has_web_channel', 'has_email_channel', 'has_mobile_channel', 'has_social_channel'
])

Feature engineering completed.
New columns added: ['membership_duration_months', 'age_group', 'income_group', 'has_web_channel', 'has_email_channel', 'has_mobile_channel', 'has_social_channel']


## Step 3) Export enriched dataset

> Converts group features to CSV-safe strings and saves the final dataset to `data/features_added/preprocessed_data_features_added.csv`.

In [3]:
output_dir = DATA_DIR / 'features_added'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'preprocessed_data_features_added.csv'

# Convert category columns to string labels for CSV compatibility
df['age_group'] = df['age_group'].astype(str)
df['income_group'] = df['income_group'].astype(str)

df.to_csv(output_path, index=False)

print('Saved:', output_path)
print('Final shape:', df.shape)
print('Preview of added feature columns:')
df[[
    'membership_duration_months', 'age_group', 'income_group',
    'has_web_channel', 'has_email_channel', 'has_mobile_channel', 'has_social_channel'
]].head()

Saved: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\features_added\preprocessed_data_features_added.csv
Final shape: (39826, 27)
Preview of added feature columns:


,membership_duration_months,age_group,income_group,has_web_channel,has_email_channel,has_mobile_channel,has_social_channel
0,15.14,26-35,middle,1,1,1,1
1,15.14,26-35,middle,1,1,1,1
2,3.02,46-55,middle,1,1,1,1
3,6.50,36-45,lower_middle,1,1,0,0
4,6.50,36-45,lower_middle,1,1,1,1
